# 📊 NLP Sentiment Analysis — Exploration Notebook
Use this notebook to explore the dataset, model, and results.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/reviews.csv')
print(df.shape)
df.head(10)

In [ ]:
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df['label_name'] = df['label'].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['label_name'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#f39c12','#2ecc71'])
axes[0].set_title('Label Distribution'); axes[0].set_xlabel('')

# Text length distribution
df['text_len'] = df['text'].str.split().str.len()
df.groupby('label_name')['text_len'].plot(kind='hist', alpha=0.5, bins=20, ax=axes[1], legend=True)
axes[1].set_title('Text Length by Label')

plt.tight_layout(); plt.show()

## 2. Preprocessing Inspection

In [ ]:
from src.data_processing.preprocessor import TextPreprocessor, clean_text

sample = 'This is an AMAZING product!!! <b>Buy it</b> now 🎉'
print('Original:', sample)
print('Cleaned: ', clean_text(sample))

## 3. Load Trained Model & Predict

In [ ]:
import os
if os.path.exists('../src/models/sentiment_model.h5'):
    from src.inference.predictor import SentimentPredictor
    p = SentimentPredictor(
        model_path='../src/models/sentiment_model.h5',
        prep_path='../src/data_processing/tokenizer.pkl'
    )
    texts = [
        'Absolutely wonderful, love this!',
        'Mediocre at best, nothing special.',
        'Terrible! Total waste of money.',
    ]
    for t in texts:
        r = p.predict_one(t)
        print(f"{r['emoji']} [{r['label']:8s}] {r['confidence']*100:.1f}% — {t}")
else:
    print('Train the model first: python src/training/train.py')

## 4. View Training Results

In [ ]:
import json
if os.path.exists('../docs/metrics.json'):
    with open('../docs/metrics.json') as f:
        metrics = json.load(f)
    print('Test metrics:', metrics)
    
    from IPython.display import Image, display
    display(Image('../docs/training_curves.png'))
    display(Image('../docs/confusion_matrix.png'))
else:
    print('Train the model first.')